# Sambar Deer Activity Analysis — Research Journey

This notebook consolidates the experimental work from five earlier notebooks (`Basic`, `Test1`, `Final1`, `Final2`, `MakeDF`) into a single document that traces, in order, the path from initial OpenCV motion detection through a failed LSTM attempt to a working threshold-based classification pipeline deployed on a live camera feed.

Each section builds on the last. Look for **Reflection** blocks for the takeaway at each pivot.

## Project context

Goal: detect *unusual motion* in Formosan Sambar deer camera-trap footage. Behaviour clips are labelled in Chinese filenames under `data/labeled/`. They are grouped into Normal / Aggressive / Passive (N / A / P) under `data/raw/水鹿影片/水路剪輯/{N,A,P}/`.

The two distinct programs in the system are:
1. **Deer classification** (this notebook + the original five): an offline pipeline that learns activity thresholds from labelled clips.
2. **Live scope detection** (`src/live_scope_detection/capture_process.py`): a real-time tkinter+OpenCV app where the user draws a region of interest on a camera feed, the program classifies motion as `Aggressive` / `Normal` / `Passive` using thresholds learned here, and pushes a LINE alert on `Aggressive`.

The legacy notebooks are preserved in `archive/legacy_notebooks/` with their original execution outputs.

## 0. Setup

All paths are relative to this notebook (`src/deer_classification/`). The data layout sits two directories up:

```
../../data/labeled/                       24 behaviour-tagged clips
../../data/raw/水鹿影片/                  raw deer videos
../../data/raw/水鹿影片/水路剪輯/{N,A,P}/  N/A/P-grouped subset
../../data/tables/                        xlsx outputs from this pipeline
../../models/                             trained classifier
```

In [ ]:
import os
import time
import cv2
import numpy as np
import pandas as pd

DATA_LABELED = "../../data/labeled"
DATA_RAW_VIDEO = "../../data/raw/水鹿影片/水鹿1.mp4"
DATA_TABLES = "../../data/tables"

STATUS_FOLDERS = {
    "Normal":     "../../data/raw/水鹿影片/水路剪輯/N",
    "Aggressive": "../../data/raw/水鹿影片/水路剪輯/A",
    "Passive":    "../../data/raw/水鹿影片/水路剪輯/P",
}

## 1. Proof of concept — live camera motion detection

Starting point: take a webcam feed, detect anything that moves, draw a bounding box around it. The technique is classic background subtraction:

1. Maintain a running average frame `avg` (`cv2.accumulateWeighted`).
2. For each new frame, blur it, subtract from `avg`, threshold the difference.
3. Run `findContours` on the threshold mask, draw a bounding box around any contour above an area threshold.

This works for *any* moving object — it has no notion of "deer" — but it's the foundation everything else is built on.

*Heads-up*: in OpenCV 4.x `cv2.findContours` returns a 2-tuple. The earliest run of this notebook crashed because it expected the OpenCV-3.x 3-tuple form. The version below is the corrected one.

In [ ]:
cap = cv2.VideoCapture(0)

width, height = 1280, 960
cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)

ret, frame = cap.read()
avg = cv2.blur(frame, (4, 4))
avg_float = np.float32(avg)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    blur = cv2.blur(frame, (4, 4))
    diff = cv2.absdiff(avg, blur)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 25, 255, cv2.THRESH_BINARY)

    kernel = np.ones((5, 5), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel, iterations=2)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)

    cnts, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in cnts:
        if cv2.contourArea(c) < 2500:
            continue
        (x, y, w, h) = cv2.boundingRect(c)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.drawContours(frame, cnts, -1, (0, 255, 255), 2)

    cv2.imshow('frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    cv2.accumulateWeighted(blur, avg_float, 0.01)
    avg = cv2.convertScaleAbs(avg_float)

cap.release()
cv2.destroyAllWindows()

## 2. Apply to a recorded video and quantify activity

From PoC to measurement: instead of drawing rectangles, **count the non-zero pixels** in the threshold mask each frame. That count is a one-number proxy for "how much motion is in this frame." Sum it across the whole video to get a total activity score.

In [ ]:
cap = cv2.VideoCapture(DATA_RAW_VIDEO)
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open {DATA_RAW_VIDEO}")

total_nonzero = 0
ret, frame = cap.read()
avg = cv2.blur(frame, (4, 4))

start = time.time()
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    blur = cv2.blur(frame, (4, 4))
    diff = cv2.absdiff(avg, blur)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 25, 255, cv2.THRESH_BINARY)
    kernel = np.ones((5, 5), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel, iterations=2)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
    total_nonzero += np.count_nonzero(thresh)

cap.release()
print(f"Run time: {time.time() - start:.2f} s  |  Total activity: {total_nonzero}")
# First run on 水鹿1.mp4: 5.66 s, total activity 48,629,694.

Same algorithm but emitting per-frame counts into a DataFrame, ready for downstream analysis:

In [ ]:
cap = cv2.VideoCapture(DATA_RAW_VIDEO)
activity_counts = []

ret, frame = cap.read()
avg = cv2.blur(frame, (4, 4))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    blur = cv2.blur(frame, (4, 4))
    diff = cv2.absdiff(avg, blur)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 25, 255, cv2.THRESH_BINARY)
    kernel = np.ones((5, 5), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel, iterations=2)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
    activity_counts.append(np.count_nonzero(thresh))

cap.release()
df_single = pd.DataFrame({"Activity_Count": activity_counts})
print(df_single.tail())
# Tail values from first run: ~85,000–86,000 per frame, 599 frames total.

## 3. Build a labelled dataset across N / A / P clips

The deer footage was hand-grouped into three folders:
- **Normal (N)**: walking, grooming, watching
- **Aggressive (A)**: lots of fast motion, chases, startle
- **Passive (P)**: resting, slow eating, lying down

Iterate every `.mp4` in each folder, run the per-frame motion detector, append `(Activity_Count, Status)` to one big table. Output: `activity_data_all.xlsx`. Roughly 25 clips, 599+ frames each → ~15k rows.

In [ ]:
all_activity_data = {"Activity_Count": [], "Status": []}

for status, folder_path in STATUS_FOLDERS.items():
    print(f"Status: {status}  folder: {folder_path}")
    video_paths = [os.path.join(folder_path, f)
                   for f in os.listdir(folder_path) if f.endswith('.mp4')]

    for video_path in video_paths:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"  skip (cannot open): {video_path}")
            continue

        ret, frame = cap.read()
        avg = cv2.blur(frame, (4, 4))

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            blur = cv2.blur(frame, (4, 4))
            diff = cv2.absdiff(avg, blur)
            gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
            _, thresh = cv2.threshold(gray, 25, 255, cv2.THRESH_BINARY)
            kernel = np.ones((5, 5), np.uint8)
            thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel, iterations=2)
            thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
            all_activity_data["Activity_Count"].append(np.count_nonzero(thresh))
            all_activity_data["Status"].append(status)
        cap.release()

df_all = pd.DataFrame(all_activity_data)
df_all.to_excel(f"{DATA_TABLES}/activity_data_all.xlsx", index=False)
print(f"Saved: {DATA_TABLES}/activity_data_all.xlsx  ({len(df_all)} rows)")

## 4. First attempt: LSTM supervised learning

Train an LSTM to map a frame's `Activity_Count` → its `Status`. Why LSTM? The first instinct was "video → sequence model." That was the wrong instinct: each row in the table is a single frame, not a sequence, so the LSTM is operating over a length-1 input. With one feature and one timestep, it has no temporal signal to learn from.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

df = pd.read_excel(f"{DATA_TABLES}/activity_data_all.xlsx")
df["Status"] = df["Status"].map({"Normal": 1, "Aggressive": 2, "Passive": 0})

X = df[["Activity_Count"]].values
y = df["Status"].values
X_scaled = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = Sequential([
    LSTM(units=50, input_shape=(X_train.shape[1], 1)),
    Dense(units=3, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(
    X_train.reshape((X_train.shape[0], X_train.shape[1], 1)), y_train,
    epochs=10, batch_size=16, validation_split=0.2,
)
loss, acc = model.evaluate(X_test.reshape((X_test.shape[0], X_test.shape[1], 1)), y_test)
print(f"Test accuracy: {acc:.3f}")
# Achieved: 0.504. Slightly better than random (0.333) but not usable.

Tried again with two stacked LSTM layers + dropout + 50 epochs:

In [ ]:
from tensorflow.keras.layers import Dropout
from sklearn.metrics import mean_squared_error, f1_score

model = Sequential([
    LSTM(units=50, input_shape=(X_train.shape[1], 1), return_sequences=True),
    Dropout(0.2),
    LSTM(units=50),
    Dropout(0.2),
    Dense(units=3, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(
    X_train.reshape((X_train.shape[0], X_train.shape[1], 1)), y_train,
    epochs=50, batch_size=16, validation_split=0.2,
)
y_pred = np.argmax(
    model.predict(X_test.reshape((X_test.shape[0], X_test.shape[1], 1))), axis=1,
)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
f1 = f1_score(y_test, y_pred, average="weighted")
print(f"RMSE: {rmse:.3f}  F1: {f1:.3f}")
# Achieved: RMSE 0.927, F1 0.552, val_accuracy plateaued ~0.58.

**Reflection.** A single scalar `Activity_Count` simply doesn't carry enough information to separate three classes — different behaviours can produce the same motion magnitude (e.g. a deer pacing slowly vs. resting and twitching). Adding capacity (more layers, dropout, more epochs) didn't fix it because the bottleneck is the feature, not the model. Time to question the framing.

## 5. Try a coarser time granularity

Hypothesis: maybe per-frame is too noisy and the model needs *temporal context*. Aggregate every 30 frames (≈ 1 second at 25 fps) into a single mean activity number, retrain. If the LSTM sees less noise per row, maybe it learns better.

In [ ]:
all_activity_data = {"Activity_Count": [], "Status": []}

for status, folder_path in STATUS_FOLDERS.items():
    print(f"Status: {status}")
    video_paths = [os.path.join(folder_path, f)
                   for f in os.listdir(folder_path) if f.endswith('.mp4')]

    for video_path in video_paths:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            continue
        ret, frame = cap.read()
        avg = cv2.blur(frame, (4, 4))
        frame_count = 0
        activity_sum = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_count % 30 == 0 and frame_count != 0:
                all_activity_data["Activity_Count"].append(activity_sum / 30)
                all_activity_data["Status"].append(status)
                activity_sum = 0

            blur = cv2.blur(frame, (4, 4))
            diff = cv2.absdiff(avg, blur)
            gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
            _, thresh = cv2.threshold(gray, 25, 255, cv2.THRESH_BINARY)
            kernel = np.ones((5, 5), np.uint8)
            thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel, iterations=2)
            thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
            activity_sum += np.count_nonzero(thresh)
            frame_count += 1
        cap.release()

df_persec = pd.DataFrame(all_activity_data)
df_persec.to_excel(f"{DATA_TABLES}/activity_data_all_persec.xlsx", index=False)
print(f"Saved {DATA_TABLES}/activity_data_all_persec.xlsx ({len(df_persec)} rows)")

In [ ]:
# Same LSTM architecture, retrained on the per-second table.
df = pd.read_excel(f"{DATA_TABLES}/activity_data_all_persec.xlsx")
df["Status"] = df["Status"].map({"Normal": 1, "Aggressive": 2, "Passive": 0})

X = df[["Activity_Count"]].values
y = df["Status"].values
X_scaled = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = Sequential([
    LSTM(units=50, input_shape=(X_train.shape[1], 1), return_sequences=True),
    Dropout(0.2),
    LSTM(units=50),
    Dropout(0.2),
    Dense(units=3, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(
    X_train.reshape((X_train.shape[0], X_train.shape[1], 1)), y_train,
    epochs=50, batch_size=16, validation_split=0.2,
)
loss, acc = model.evaluate(X_test.reshape((X_test.shape[0], X_test.shape[1], 1)), y_test)
print(f"Test accuracy: {acc:.3f}")
# Achieved: 0.314. Worse than per-frame, and worse than random.

**Reflection.** Per-second averaging shrank the dataset to ~330 rows without giving the LSTM the temporal context it would need to actually exploit sequences. Lower data, same input shape, no extra information per row → worse model. The right move is not to keep tweaking time granularity — it's to **change the framing**.

One more aggregation, this time per video (one row = one clip's total activity), kept for reference but used by the next section instead of training a model on it:

In [ ]:
rows = []
for status, folder_path in STATUS_FOLDERS.items():
    for f in os.listdir(folder_path):
        if not f.endswith('.mp4'):
            continue
        video_path = os.path.join(folder_path, f)
        cap = cv2.VideoCapture(video_path)
        total_nonzero = 0
        ret, frame = cap.read()
        avg = cv2.blur(frame, (4, 4))
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            blur = cv2.blur(frame, (4, 4))
            diff = cv2.absdiff(avg, blur)
            gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
            _, thresh = cv2.threshold(gray, 25, 255, cv2.THRESH_BINARY)
            kernel = np.ones((5, 5), np.uint8)
            thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel, iterations=2)
            thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
            total_nonzero += np.count_nonzero(thresh)
        cap.release()
        rows.append({"Total_Activity": total_nonzero, "Status": status, "Video_Path": video_path})

df_pervideo = pd.DataFrame(rows)
df_pervideo.to_excel(f"{DATA_TABLES}/total_activity_by_video.xlsx", index=False)
print(f"Saved {DATA_TABLES}/total_activity_by_video.xlsx ({len(df_pervideo)} rows)")

## 6. Pivot — threshold-based classification

New approach: **don't train a classifier; learn thresholds.**

Two changes from earlier sections:
1. **Different feature.** Instead of "count of pixels above an absolute threshold", use a *relative* per-frame difference: `sum(|frame - prev| / prev)`. This downweights regions that are already bright/active and emphasises actual change. Numerically: a single number per frame, but on a different distribution from the count.
2. **Different classifier.** Compute the 30th and 70th percentile of all frame differences across the whole labelled set. Each frame's classification is then:
   - `diff ≥ 70th pct` → **Aggressive**
   - `30th pct ≤ diff < 70th pct` → **Normal**
   - `diff < 30th pct` → **Passive**

Then summarise per video: what fraction of its frames fell into each class? If one class accounts for ≥ 75% of frames the video gets that label; otherwise it's labelled `Normal` (the catch-all).

In [ ]:
def calculate_frame_difference(prev_frame, current_frame):
    diff = cv2.absdiff(prev_frame, current_frame)
    epsilon = 1e-10
    return np.sum(diff / (prev_frame + epsilon))

def process_video(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_frame = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    differences = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        current_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        differences.append(calculate_frame_difference(prev_frame, current_frame))
        prev_frame = current_frame
    cap.release()
    return differences

def calculate_thresholds(differences, low_pct=30, high_pct=70):
    return np.percentile(differences, low_pct), np.percentile(differences, high_pct)

def classify_activities(differences, low, high):
    out = []
    for d in differences:
        if d >= high:
            out.append(('Aggressive', d))
        elif d >= low:
            out.append(('Normal', d))
        else:
            out.append(('Passive', d))
    return out

def process_folder(folder_path, low_pct=30, high_pct=70):
    all_differences = []
    for fn in os.listdir(folder_path):
        if fn.lower().endswith(('.mp4', '.avi', '.mov')):
            all_differences.extend(process_video(os.path.join(folder_path, fn)))

    low, high = calculate_thresholds(all_differences, low_pct, high_pct)
    rows = []
    for fn in os.listdir(folder_path):
        if fn.lower().endswith(('.mp4', '.avi', '.mov')):
            video_path = os.path.join(folder_path, fn)
            for activity, d in classify_activities(process_video(video_path), low, high):
                rows.append({'video': fn, 'activity': activity, 'difference': d})
    return rows, low, high

results, low, high = process_folder(DATA_LABELED, 30, 70)
pd.DataFrame(results).to_excel(f"{DATA_TABLES}/output_results.xlsx", index=False)
print(f"Thresholds: low={low:.3e}  high={high:.3e}")
print(f"Saved {DATA_TABLES}/output_results.xlsx")

In [ ]:
# Per-video activity-class percentages.
data = pd.read_excel(f"{DATA_TABLES}/output_results.xlsx")
activity_counts = data.groupby('video')['activity'].value_counts().unstack(fill_value=0)
activity_percentages = (
    activity_counts.div(activity_counts.sum(axis=1), axis=0) * 100
).round(0).astype(int)
activity_percentages

In [ ]:
# Apply the >75% rule to assign one main_activity per video.
main_activity = activity_percentages.idxmax(axis=1)
main_activity_pct = activity_percentages.max(axis=1)
main_activity[main_activity_pct <= 75] = 'Normal'

summary = pd.DataFrame({
    'video': main_activity.index,
    'main_activity': main_activity.values,
    'Aggressive': activity_percentages['Aggressive'],
    'Normal':     activity_percentages['Normal'],
    'Passive':    activity_percentages['Passive'],
})
summary.to_excel(f"{DATA_TABLES}/new_data.xlsx", index=False)
summary

**Reflection.** The threshold approach correctly flagged the high-energy clips (`受水驚嚇` 87% Aggressive, `繞圈走動` 87%, `走來走去` 69%) and the rest-heavy clips (`走過來給鹿舔` 80% Passive, `休息2` 79%, `休息＋人經過轉頭` 77%). Most clips fall in the `Normal` bucket — including ambiguous ones like `舔自己` and `亂走亂舔` where motion is real but moderate. Without training a model, the algorithm produced a reasonable labelling. The pivot worked.

## 7. Sanity-check with K-means clustering

If the labelling is meaningful, an unsupervised algorithm given the same features should re-discover similar groups. Run K-means with `k=3` on each video's `(Aggressive%, Normal%, Passive%)` vector and see whether clusters align with the threshold-based labels.

In [ ]:
from sklearn.cluster import KMeans

scaled = StandardScaler().fit_transform(activity_percentages)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(scaled)

out = activity_percentages.copy()
out['cluster'] = kmeans.labels_
out.to_excel(f"{DATA_TABLES}/video_activity_clustering.xlsx")
out[['cluster']]

**Reflection.** K-means produced clusters that mostly align with the threshold labels: a high-Aggressive cluster (受水驚嚇, 繞圈走動, 走來走去, 走來走去＋搖尾巴), a high-Passive cluster (走過來給鹿舔, 舔自己, 趴下動作, etc.), and a `Normal`-dominant catch-all. Disagreements are interesting (`互舔` clusters with the Normal-dominant group despite being grouped under N originally) but don't undermine the approach. Two independent methods reached compatible groupings → the underlying signal is real.

## 8. Failed experiment — Gaussian thresholds

Side experiment: what if we set thresholds at `mean ± 1σ` instead of percentiles? Theoretically more principled — but only if the diff distribution is approximately Gaussian. It isn't.

In [ ]:
def calc_mean_diff(prev, cur):
    return np.mean(cv2.absdiff(prev, cur))

def process_video_mean(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, prev = cap.read()
    prev = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
    diffs = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cur = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        diffs.append(calc_mean_diff(prev, cur))
        prev = cur
    cap.release()
    return diffs

all_diffs = []
for fn in os.listdir(DATA_LABELED):
    if fn.lower().endswith(('.mp4', '.avi', '.mov')):
        all_diffs.extend(process_video_mean(os.path.join(DATA_LABELED, fn)))

mean = np.mean(all_diffs)
std  = np.std(all_diffs)
low  = mean - 1.0 * std
high = mean + 1.0 * std
print(f"mean={mean:.3f}  std={std:.3f}  low={low:.3f}  high={high:.3f}")
# Standard deviation in the original run: 1.645.

**Reflection.** Two problems:
1. The diff distribution is right-skewed (long tail of high-motion frames), so `mean - 1σ` falls below zero for many videos and the **Passive** class effectively disappears — leaving a 2-class output instead of 3.
2. `mean ± 1σ` puts roughly 68% of frames in the central bin — wider than the percentile method's 40%-wide central bin. Less discrimination, not more.

Going back to percentile thresholds, but tighter (25/75 instead of 30/70) for sharper separation.

## 9. Final pipeline — tuned thresholds + reusable functions

The pipeline below is the one the live deployment uses. Differences from §6:
- Tighter percentiles: 25/75 instead of 30/70
- Frame difference is the raw absolute pixel sum (no division by `prev`) — empirically equivalent on this data and faster
- Thresholds are explicitly serialised so the live program can reuse them

In [ ]:
def frame_difference_abs(prev_frame, current_frame):
    return np.sum(cv2.absdiff(prev_frame, current_frame))

def video_differences(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, prev = cap.read()
    if not ret:
        return []
    prev = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
    diffs = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cur = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        diffs.append(frame_difference_abs(prev, cur))
        prev = cur
    cap.release()
    return diffs

def calibrate_thresholds(folder_path, low_pct=25, high_pct=75):
    all_diffs = []
    for fn in os.listdir(folder_path):
        if fn.lower().endswith(('.mp4', '.avi', '.mov')):
            all_diffs.extend(video_differences(os.path.join(folder_path, fn)))
    return (
        float(np.percentile(all_diffs, low_pct)),
        float(np.percentile(all_diffs, high_pct)),
    )

def classify_diffs(diffs, low, high):
    return [
        'Aggressive' if d >= high else 'Normal' if d >= low else 'Passive'
        for d in diffs
    ]

low, high = calibrate_thresholds(DATA_LABELED, 25, 75)
thresholds_df = pd.DataFrame({'low_threshold': [low], 'high_threshold': [high]})
thresholds_df.to_excel(f"{DATA_TABLES}/thresholds.xlsx", index=False)
thresholds_df
# Original calibration: low=707953.75   high=1950303.0

In [ ]:
# Single-video sanity check: classify every frame of '受水驚嚇.mp4' (a clearly aggressive clip).
test_video = f"{DATA_LABELED}/受水驚嚇.mp4"
diffs = video_differences(test_video)
labels = classify_diffs(diffs, low, high)
df_one = pd.DataFrame({
    'video': os.path.basename(test_video),
    'frame': range(1, len(diffs) + 1),
    'difference': diffs,
    'activity': labels,
})
summary_one = (
    df_one.groupby('video')['activity'].value_counts().unstack(fill_value=0)
         .div(len(df_one)) * 100
).round(0).astype(int)
summary_one
# Original run: 96% Aggressive, 4% Normal, 0% Passive — matches the human label.

**Reflection.** End-to-end the threshold pipeline labelled the test clip 96% Aggressive — the human label was Aggressive. Without any model fitting, this is the working classifier.

Why the threshold approach beats the LSTM here:
- The class boundaries are *defined* in terms of the feature, not learned
- Calibration is a one-shot computation across the labelled set
- It generalises to a single live video (compute its diffs, apply the saved thresholds) without retraining
- It is interpretable: a frame's verdict comes with the exact numerical reason

## 10. Live deployment

The thresholds calibrated above feed the live program at `src/live_scope_detection/capture_process.py`. That script:
1. Opens the default webcam in a tkinter window
2. Lets the user drag a rectangle to define a region of interest
3. Computes the same `frame_difference_abs` *only inside the ROI*
4. Buffers 5 seconds of diffs, classifies the buffer's distribution as Aggressive / Normal / Passive (one tag per 5-second window using a >75% rule similar to §6)
5. On `Aggressive`, sends a LINE notification (token in `.env`)

The legacy in-notebook prototype that became `capture_process.py` lives commented out at the bottom of `archive/legacy_notebooks/Final2.ipynb`.

## 11. Conclusion

**What worked.** A surprisingly simple pipeline: relative-magnitude pixel diffs → percentile thresholds calibrated on labelled clips → frame-level classification → per-video aggregation with a >75% rule. Validated by an unsupervised K-means cluster check.

**What didn't, and why.** The instinct to throw an LSTM at the problem failed because the input was a single scalar per timestep — there was no sequence for the LSTM to model. Going coarser (per-second) made it worse by shrinking the dataset without adding information. Gaussian-based thresholds collapsed under a skewed distribution and erased the Passive class.

**What's next.**
- The LINE Notify transport is dead as of 2025-03-31. The live program needs a new alert channel (LINE Messaging API, Discord webhook, or Telegram bot).
- The `data/raw/水鹿影片/水路剪輯/` N/A/P split currently re-uses the labelled clips from `data/labeled/`. Adding fresh field footage would let me test whether the calibrated thresholds transfer.
- A stricter evaluation — confusion matrix between predicted main_activity and the folder labels — would put a number on "how well does this work" instead of relying on spot-checks.